# FeFET 直流 Id–Vg (SMU) · qzb 1T · inline 看图 (51)

> 2026-06-11 建。**DC 转移特性 = SMU 干的活,不碰 WGFMU。** 走 canonical
> `fefetlab.measurements.dc.dc_sweep_api.DCSweepAPI`(单一来源,非手搓)。
> 对应 PPT 第6页【一、直流测试条件】：Vg −4→4V,Vd 0.1V,Vs 0V。

## ⚠️ 物理 / 安全 必读
- **Vg ±4V 已超过写/擦阈值**(编程 1.5–3.8V / 擦除 −3~−4V) → 这条 DC 扫描**会在扫的过程中写/擦 FeFET**,
  得到的是 **回滞记忆窗 Id–Vg 环**(FeFET 标准 DC 表征)。**双扫(−4→+4→−4)才看得到完整环。**
- **合规电流(compliance)**保护器件:本本默认 10 µA(> 编程 200 nA,且限制故障电流)。读数走 SMU 自动量程,nA 可分辨。
- **SMU 接线**:本本假设器件的 G/D/S 接到 SMU 通道(默认 4/5/6)。**若你的器件只接到 WGFMU(经 RSU),SMU 量不到** ——
  先确认探针/夹具把 G/D/S 引到了 SMU。

## DC 与 WGFMU 引擎门的关系(现状)
DC 这条腿**暂未并入 `ProtocolEngine` 引擎门**(那是 M1 "DC 注册"待办)。本本直接调 DCSweepAPI,
但已把输出落到 **`runs/<device>/<die>/live/<ts>_idvg_sweep/`** 两级布局,与 WGFMU 数据归集一致。

## 用法
1. 测试机 `D:\test\B1500` 根目录起 `jupyter lab`,打开本本。
2. 改【配置】cell：器件身份 / SMU 通道 / 扫描范围 / 合规。
3. **先跑【dry 预览】cell**(不开硬件,核对扫描点/通道/落盘路径),再跑【执行】cell 上真机。

In [ ]:
# ===================== 【配置】改这里 =====================
DEVICE_ID   = "qzb_sy2"          # 批次/自命名 → 顶层归集文件夹
GEOMETRY    = "L300W300"         # 这颗的几何 沟长×沟宽
SERIAL      = "1"                # ← 序号 / die 号
DEVICE_TYPE = "FeFET"
OPERATOR    = "yhzang"

# SMU 通道:Gate=SMU2(slot5,经 RSU 与 WGFMU202 共栅)=5;Drain/Source 在 SMU1(slot4)/SMU3(slot6)
CH_G, CH_D, CH_S = 5, 4, 6       # ← drain/source 哪个 SMU 自行确认(默认 D=SMU1/4, S=SMU3/6)

# DC Id-Vg 扫描(PPT 第6页直流条件)
VG_START, VG_STOP, VG_STEP = -4.0, 4.0, 0.1   # Vg 扫描 V
DUAL      = True                 # True=双扫 −4→+4→−4(看回滞环);False=单扫
VD_FIXED  = 0.1                  # Vd V
VS_FIXED  = 0.0                  # Vs V
COMPLIANCE_A = 10e-6             # 各通道合规电流 A(保护;> 编程 200 nA)

VISA_RESOURCE = ""               # 留空=自动探测/按 B1500_VISA_ADDR 环境变量;或填 "GPIB1::17::INSTR"
# =========================================================
print(f"器件 {DEVICE_ID}/{GEOMETRY}#{SERIAL}  SMU G/D/S={CH_G}/{CH_D}/{CH_S}")
print(f"Vg {VG_START}->{VG_STOP} step {VG_STEP} dual={DUAL}  Vd={VD_FIXED} Vs={VS_FIXED}  Icomp={COMPLIANCE_A}A")

In [ ]:
import sys, os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "fefetlab").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.measurements.dc.dc_sweep_api import DCSweepAPI
from fefetlab.measurements.dc.config import DCSweepConfig, DCChannelConfig
from fefetlab.orchestration.core import ExperimentContext


def build_vg_points():
    # 构造扫描点;DUAL 时 −4→+4→−4(去掉重复峰值)
    fwd = np.round(np.arange(VG_START, VG_STOP + VG_STEP / 2, VG_STEP), 6)
    pts = list(fwd)
    if DUAL:
        pts += list(fwd[::-1][1:])
    return [float(v) for v in pts]


def build_config():
    # SMU 三通道合规 + 自动量程(vrange=0)。delay/平均沿用 DC 默认。
    return DCSweepConfig(channels={
        "G": DCChannelConfig(channel=CH_G, vrange=0, compliance=COMPLIANCE_A, role="G"),
        "D": DCChannelConfig(channel=CH_D, vrange=0, compliance=COMPLIANCE_A, role="D"),
        "S": DCChannelConfig(channel=CH_S, vrange=0, compliance=COMPLIANCE_A, role="S"),
    }, delay_s=0.2, av_count=10)


def export_dir():
    # 落进两级布局 runs/<device>/<die>/live/(DCSweepAPI 再追加 <ts>_idvg_sweep)
    ctx = ExperimentContext(root=ROOT, device_id=DEVICE_ID, geometry=GEOMETRY,
                            serial=SERIAL, live=True)
    return ROOT / "runs" / ctx.device_slug / ctx.die_slug / "live"


def resolve_resource():
    if VISA_RESOURCE.strip():
        return VISA_RESOURCE.strip()
    env = (os.environ.get("B1500_VISA_ADDR") or "").strip()
    if env:
        return env
    from fefetlab.measurements.wgfmu.setup_helpers import autodetect_visa_addr
    return autodetect_visa_addr("B1500")


def plot_idvg_dc(df, title=None):
    # |Id|-Vg 回滞(前向/反向分色) + |Ig| 栅漏监看
    vg = df["vg_set"].to_numpy()
    idd = df["id_A"].abs().to_numpy()
    turn = int(np.argmax(vg))
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(vg[:turn + 1], idd[:turn + 1], "o-", ms=3, color="#1f77b4", label="forward −→+")
    if turn < len(vg) - 1:
        ax.plot(vg[turn:], idd[turn:], "s-", ms=3, color="#d62728", label="reverse +→−")
    ax.set_yscale("log"); ax.set_xlabel("Vg (V)"); ax.set_ylabel("|Id| (A)")
    ax.set_title(title or f"DC Id-Vg 回滞  {DEVICE_ID} {GEOMETRY}#{SERIAL}")
    ax.legend(); ax.grid(True, which="both", alpha=.3); plt.show()

    fig2, ax2 = plt.subplots(figsize=(6.5, 2.6))
    ax2.plot(vg, df["ig_A"].abs().to_numpy(), "r.-")
    ax2.set_yscale("log"); ax2.set_xlabel("Vg (V)"); ax2.set_ylabel("|Ig| (A)")
    ax2.set_title("栅漏监看 |Ig|"); ax2.grid(alpha=.3); plt.show()


print("DC 工具就绪。")

## dry 预览（不开硬件，核对扫描点 / 通道 / 落盘路径）

In [ ]:
vg_points = build_vg_points()
print(f"扫描点数: {len(vg_points)}  首{vg_points[:3]} ... 峰{max(vg_points)} ... 末{vg_points[-3:]}")
print(f"SMU 通道 G/D/S = {CH_G}/{CH_D}/{CH_S}   合规 {COMPLIANCE_A} A")
print(f"落盘到: {export_dir()}\\<时间戳>_idvg_sweep\\")
print("确认无误后再跑下面【执行】cell（会开 VISA 驱动真机 SMU）。")

## 执行（开 VISA，驱动真机 SMU 扫 Id–Vg）
器件已落针、SMU 接线确认后再跑。

In [ ]:
resource = resolve_resource()
print(f"VISA: {resource}")
visa_cfg = VisaConfig(resource=resource, timeout_ms=30000,
                      write_termination="\r\n", read_termination="\r\n", send_end=True)

with VisaSession(visa_cfg) as session:
    api = DCSweepAPI(session, ch_g=CH_G, ch_d=CH_D, ch_s=CH_S,
                     config=build_config(), export_dir=export_dir())
    result = api.run_idvg_sweep(vg_points=vg_points, vd_fixed=VD_FIXED,
                                vs_fixed=VS_FIXED, auto_export=True, verbose=True)

df = result["df"]
print("保存到:", result["run_dir"])
df[["vg_set", "vd_set", "id_A", "ig_A", "is_A", "status"]].head()

## 看图（Id–Vg 回滞环 + 栅漏监看）

In [ ]:
plot_idvg_dc(df)